# Private-Credit Explorer — a developer's notebook

Your rough sandbox for the *"Gauging bank private-credit exposure"* piece. Every
intermediate calculation is visible here so you can be **human-in-the-loop**: run a
cell, read the table, change something, re-run.

**How to use (in Cursor):**
1. Open this file. Top-right, click **Select Kernel** -> pick the `.venv` Python.
2. Click a cell and press **Shift+Enter** to run it and move to the next.
3. Edit any cell and re-run — nothing here touches the published site.

Reads the *same real data* the article uses: `analysis/outputs/call_report_ndfi_panel.parquet`.

In [ ]:
from pathlib import Path
import pandas as pd

here = Path.cwd()
candidates = [here / "analysis/outputs/call_report_ndfi_panel.parquet",
              here / "../outputs/call_report_ndfi_panel.parquet"]
PANEL = next(p for p in candidates if p.exists())

panel = pd.read_parquet(PANEL)
print("rows:", len(panel), "| quarters:", sorted(panel['quarter'].dt.date.unique()))
panel.head()

## 1. The raw material — one row per bank *subsidiary*

Each insured bank files its own quarterly Call Report. The two fields that matter:
- `LNNDEPC` — loans to non-depository financial institutions (**NDFI**), $thousands
- `RBCT1J` — **Tier 1 capital**, $thousands

In [ ]:
latest = panel["quarter"].max()
cur = panel[panel["quarter"] == latest].copy()
print("latest quarter:", latest.date(), "| banks reporting NDFI:", len(cur))

view = cur[["NAME", "NAMEHCR", "LNNDEPC", "RBCT1J", "ASSET"]].copy()
view[["LNNDEPC","RBCT1J","ASSET"]] = (view[["LNNDEPC","RBCT1J","ASSET"]] / 1e6).round(1)
view.sort_values("LNNDEPC", ascending=False).head(8)

## 2. The market lens — just sum the one column

The "$1.5 trillion market" headline is one `.sum()`.

In [ ]:
total_bn = cur["LNNDEPC"].sum() / 1e6
top10 = cur.nlargest(10, "LNNDEPC")["LNNDEPC"].sum() / 1e6
print(f"Total drawn NDFI:  ${total_bn:,.0f}B  (${total_bn/1000:.2f}T)")
print(f"Top-10 banks hold: {top10/total_bn:.0%} of it")

## 3. The subsidiary problem — Morgan Stanley files *twice*

Some firms lend through several bank subsidiaries. Don't roll them up and one company
shows as several points with a wrong ratio.

In [ ]:
ms = cur.loc[cur["NAMEHCR"].str.contains("MORGAN STANLEY"),
             ["NAME", "LNNDEPC", "RBCT1J"]].copy()
ms[["LNNDEPC","RBCT1J"]] = (ms[["LNNDEPC","RBCT1J"]] / 1e6).round(1)
ms

## 4. Roll subsidiaries up to the holding company

Group by the top-tier company (`NAMEHCR`) and sum. One firm = one row.

In [ ]:
grp = (cur.assign(entity=cur["NAMEHCR"].str.strip().replace("", pd.NA).fillna(cur["NAME"]))
          .groupby("entity")
          .agg(ndfi=("LNNDEPC","sum"), tier1=("RBCT1J","sum"),
               assets=("ASSET","sum"), loans=("LNLSNET","sum"))
          .reset_index())

ms_holdco = grp[grp["entity"].str.contains("MORGAN STANLEY")]
(ms_holdco[["ndfi","tier1"]] / 1e6).round(1)

## 5. The concentration metric — NDFI ÷ Tier 1 capital

This one division is what the bubble chart is built on.

In [ ]:
grp["ndfi_to_tier1"] = grp["ndfi"] / grp["tier1"]
grp["ndfi_bn"] = grp["ndfi"] / 1e6

ranking = (grp[grp["ndfi_bn"] >= 10]
           .sort_values("ndfi_to_tier1", ascending=False)
           [["entity", "ndfi_bn", "ndfi_to_tier1"]]
           .head(15).reset_index(drop=True))
ranking["ndfi_bn"] = ranking["ndfi_bn"].round(1)
ranking["ndfi_to_tier1"] = (ranking["ndfi_to_tier1"]*100).round(0).astype(int).astype(str) + "%"
ranking

## 6. A quick chart — scale vs. concentration

A rough matplotlib version of the article's bubble chart.

In [ ]:
import matplotlib.pyplot as plt

g = grp[grp["ndfi_bn"] >= 1].copy()
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(g["ndfi_bn"], g["ndfi_to_tier1"], s=g["assets"]/1e4,
           alpha=0.4, color="#8CA0B4", edgecolor="#61758A")
ax.set_xscale("log")
ax.axhline(1.0, ls="--", color="#B4AD9E", lw=1)
ax.set_xlabel("NDFI loans, $bn (log)")
ax.set_ylabel("NDFI / Tier 1 capital")
ax.set_title("Scale is not concentration")
for name in ["GOLDMAN SACHS GROUP INC THE", "FIRST CITIZENS BANCSHARES INC", "TRUIST FINANCIAL CORP"]:
    row = g[g["entity"] == name]
    if len(row):
        ax.annotate(name.title()[:16], (row["ndfi_bn"].iloc[0], row["ndfi_to_tier1"].iloc[0]), fontsize=8)
plt.show()

## 7. Your turn — scratch space

Try (edit and re-run):
- Change `latest` in cell 2 to an **earlier quarter** and watch the total grow.
- Filter to **one bank**: `cur[cur['NAMEHCR'].str.contains('WELLS')]`.
- Try a **different ratio**: NDFI / total loans (`LNLSNET`) instead of / Tier 1.
- Sort the ranking by **dollars** instead of concentration.

Nothing here affects the article or the site — break things freely.

In [ ]:
# scratch — which banks lend the most NDFI as a share of their TOTAL loans?
grp["ndfi_to_loans"] = grp["ndfi"] / grp["loans"]
(grp[grp["ndfi_bn"] >= 10]
   .sort_values("ndfi_to_loans", ascending=False)
   [["entity", "ndfi_bn", "ndfi_to_loans"]]
   .head(10))